# Task 1 — Student Grade Analyser
**Course:** STQD6014 Data Science  
**Dataset:** `students.txt` — 50 students, 5 subjects, weighted GPA computation

**Grading scale:**
| Mark | Grade Value |
|---|---|
| 80–100 | 4.00 |
| 75–79 | 3.67 |
| 70–74 | 3.33 |
| 65–69 | 3.00 |
| 60–64 | 2.67 |
| 55–59 | 2.33 |
| 50–54 | 2.00 |
| 45–49 | 1.67 |
| 40–44 | 1.33 |
| 35–39 | 1.00 |
| < 35  | 0.00 |

In [ ]:
import pandas as pd
import numpy as np

# Load the fixed-format text file
# Skip the 3 comment lines at the top (lines starting with #)
students = pd.read_table('data/students.txt', skiprows=3, sep=',')

print(f"Dataset shape: {students.shape}")
print(f"Columns: {students.columns.tolist()}")
students.head()

In [ ]:
# Verify no missing values in mark / credit columns
marking = ['Programming','Data_Structures','Database_Systems','Web_Development','Mathematics']
credits = ['Programming_Credit','Data_Structures_Credit','Database_Systems_Credit',
           'Web_Development_Credit','Mathematics_Credit']

print("Missing values in mark columns:")
print(students[marking].isnull().sum())
print()
print("Mark ranges:")
print(students[marking].describe().round(1))

## StudentGradeAnalyzer class
Encapsulates the grading scale, GPA calculation, and student record display.  
**FIX:** Moved class definition before the first call — original notebook had the
markdown demo cell (cell 3) before the class definition (cell 4), which would raise
`NameError: name 'StudentGradeAnalyzer' is not defined`.

In [ ]:
class StudentGradeAnalyzer:
    """Compute weighted GPA and display student records from the STQD6014 dataset."""

    SUBJECTS = ['Programming', 'Data_Structures', 'Database_Systems',
                'Web_Development', 'Mathematics']
    CREDITS  = ['Programming_Credit', 'Data_Structures_Credit', 'Database_Systems_Credit',
                'Web_Development_Credit', 'Mathematics_Credit']

    def __init__(self, dataframe):
        self.df = dataframe.copy()

    # ------------------------------------------------------------------
    def mark_to_grade_point(self, mark: float) -> float:
        """Convert a percentage mark to a 4-point grade value."""
        if mark >= 80:  return 4.00
        if mark >= 75:  return 3.67
        if mark >= 70:  return 3.33
        if mark >= 65:  return 3.00
        if mark >= 60:  return 2.67
        if mark >= 55:  return 2.33
        if mark >= 50:  return 2.00
        if mark >= 45:  return 1.67
        if mark >= 40:  return 1.33
        if mark >= 35:  return 1.00
        return 0.00

    # ------------------------------------------------------------------
    def calculate_student_gpa(self, student_record) -> float:
        """Compute weighted GPA: sum(grade_value × credit) / sum(credit)."""
        marks   = student_record[self.SUBJECTS].astype(float)
        credits = student_record[self.CREDITS].astype(float)
        grade_values = marks.apply(self.mark_to_grade_point)
        gpa = (grade_values.values * credits.values).sum() / credits.values.sum()
        return round(gpa, 2)

    # ------------------------------------------------------------------
    def display_student_by_id(self, student_id: str) -> None:
        """Print a full subject breakdown and GPA for one student."""
        row = self.df[self.df['Student_ID'] == student_id]
        if row.empty:
            print(f"Student ID '{student_id}' not found.")
            return

        s = row.iloc[0]
        print(f"\nStudent Record: {s['Student_ID']}")
        print(f"Name : {s['Name']}")
        print(f"Age  : {s['Age']}\n")
        print("Subject Breakdown:")

        for subj, cred in zip(self.SUBJECTS, self.CREDITS):
            mark  = s[subj]
            gv    = self.mark_to_grade_point(mark)
            print(f"  {subj:25s}  Mark={mark:3.0f}  Credit={s[cred]}  Grade Value={gv:.2f}")

        gpa = self.calculate_student_gpa(s)
        print(f"\n  Final GPA: {gpa}")
        print("-" * 55)

    # ------------------------------------------------------------------
    def summary_stats(self) -> pd.DataFrame:
        """Return a DataFrame with each student's GPA and letter grade."""
        gpas = self.df.apply(self.calculate_student_gpa, axis=1)
        result = self.df[['Student_ID', 'Name', 'Age']].copy()
        result['GPA'] = gpas

        def letter(g):
            if g >= 3.67: return 'A'
            if g >= 3.33: return 'A-'
            if g >= 3.00: return 'B+'
            if g >= 2.67: return 'B'
            if g >= 2.33: return 'B-'
            if g >= 2.00: return 'C+'
            if g >= 1.67: return 'C'
            if g >= 1.33: return 'C-'
            if g >= 1.00: return 'D'
            return 'F'

        result['Grade'] = result['GPA'].apply(letter)
        return result.sort_values('GPA', ascending=False).reset_index(drop=True)

## Demo — display individual student records

In [ ]:
analyzer = StudentGradeAnalyzer(students)

# Demo: display specific students
analyzer.display_student_by_id("S010")
analyzer.display_student_by_id("S025")
analyzer.display_student_by_id("S111")   # should print 'not found'

## Class-wide GPA summary

In [ ]:
summary = analyzer.summary_stats()
print(f"Class GPA  — Mean: {summary['GPA'].mean():.2f}  "
      f"Max: {summary['GPA'].max():.2f}  Min: {summary['GPA'].min():.2f}")
print()
print("Top 5 students:")
print(summary.head())
print()
print("Grade distribution:")
print(summary['Grade'].value_counts().sort_index())